# TOPIC 10 – ĐO LƯỜNG RỦI RO DANH MỤC VN30 BẰNG DCC-GARCH
## Thành viên 5 – Thủy Nguyên: Euler Risk Contribution, Stress, khuyến nghị
Chạy **sau** TV4. Chạy bằng **Restart Kernel → Run All**. Thứ tự chung của cả nhóm: TV1 → TV2 → TV3 → TV4 → TV5.

| | |
|---|---|
| Đọc từ | kết quả TV1, TV2, TV3 và TV4 (`TV4_var_backtest.xlsx`, `oos_*.csv`, `tv4_settings.json`) |
| Ghi ra | `TV5_Euler_Stress/` (`TV5_euler_stress.xlsx`, `recommendations.txt`, `euler_rc_timeseries.csv`, hình) |
| Cấu hình | ô 0.2 (đường dẫn, hằng số) – **giống hệt ở cả 5 notebook**, chỉ cần sửa `PROJECT_PATH` / `OUTPUT_PATH` cho đúng máy |

Thư viện: numpy, pandas, scipy, matplotlib, openpyxl (statsmodels tuỳ chọn). Xem `README.md`.

In [ ]:
# ===================== 0.1 THƯ VIỆN =====================
import os                     # đọc biến môi trường
import re                     # biểu thức chính quy: tách dòng cổ tức / chia tách trong file Yahoo
import warnings               # tắt cảnh báo không cần thiết
import datetime as dt         # kiểu ngày tháng
from itertools import combinations   # tạo các cặp cổ phiếu
from pathlib import Path      # thao tác đường dẫn an toàn trên Windows

import numpy as np            # tính toán ma trận
import pandas as pd           # bảng dữ liệu
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats                     # phân phối Normal / t / chi-bình phương, JB
from scipy.optimize import minimize         # tối đa hoá hàm hợp lý (MLE)
from scipy.signal import lfilter            # lọc tuyến tính nhanh cho phương trình GARCH
from scipy.special import gammaln           # log Gamma cho mật độ Student-t

try:                                        # statsmodels: chỉ dùng cho ADF / KPSS (nếu có)
    from statsmodels.tsa.stattools import adfuller, kpss
    from statsmodels.tools.sm_exceptions import InterpolationWarning
    HAS_SM = True
except ImportError:
    HAS_SM = False

try:                                        # display() có sẵn trong Jupyter; chạy file .py thì thay bằng print
    display
except NameError:
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100
print("statsmodels có sẵn:", HAS_SM)

In [ ]:
# ===================== 0.2 CẤU HÌNH DÙNG CHUNG CHO CẢ 5 THÀNH VIÊN =====================
# >>> ĐƯỜNG DẪN – sửa 2 dòng này nếu đổi máy / đổi thư mục <<<
PROJECT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo"          # thư mục dự án (chứa input/, ouput/, code/)
OUTPUT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo\ouput"    # nơi lưu toàn bộ kết quả


def find_base_dir() -> Path:
    """Tìm thư mục dự án: biến môi trường PROJECT_DIR -> PROJECT_PATH -> thư mục cha của code/."""
    env = os.environ.get("PROJECT_DIR")
    if env:
        return Path(env)
    if Path(PROJECT_PATH).exists():
        return Path(PROJECT_PATH)
    cwd = Path.cwd()
    return cwd.parent if cwd.name.lower() == "code" else cwd


def pick_dir(base: Path, names) -> Path:
    """Chọn thư mục con đã tồn tại (chấp nhận Input/input)."""
    return next((base / n for n in names if (base / n).exists()), base / names[0])


BASE_DIR = find_base_dir()
INPUT_DIR = pick_dir(BASE_DIR, ["input", "Input"])
if not list(INPUT_DIR.glob("*.xlsx")):                  # dự phòng: thư mục Input cũ của Project midterm
    old_in = Path(r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\Project midterm\Input")
    if list(old_in.glob("*.xlsx")):
        INPUT_DIR = old_in
# Output: biến môi trường OUTPUT_DIR -> OUTPUT_PATH (nếu ổ D: tồn tại) -> BASE_DIR/ouput
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR") or
                  (OUTPUT_PATH if Path(OUTPUT_PATH).anchor and Path(Path(OUTPUT_PATH).anchor).exists()
                   else BASE_DIR / "ouput"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_OUT = OUTPUT_DIR                                    # mỗi thành viên một thư mục con trong ouput/
OUT = {k: ROOT_OUT / v for k, v in {"TV1": "TV1_Data_EDA", "TV2": "TV2_GARCH", "TV3": "TV3_DCC",
                                     "TV4": "TV4_VaR_Backtest", "TV5": "TV5_Euler_Stress"}.items()}
for d in OUT.values():
    d.mkdir(parents=True, exist_ok=True)

# ---- Hằng số dùng chung ----
TICKERS_ORDER = ["FPT", "HPG", "MWG", "VCB", "VHM"]    # thứ tự hiển thị (mã khác trong Input vẫn được đọc)
ALPHA = 0.05                 # mức ý nghĩa của mọi kiểm định
LB_LAGS = [5, 10, 20]        # số trễ Ljung-Box
TRADING_DAYS = 252           # số phiên / năm để năm hoá
SCALE = 100.0                # ước lượng GARCH trên lợi suất % (r × 100) cho ổn định số học
SELECT_BY = "BIC"            # tiêu chí chọn mô hình GARCH: "AIC" hoặc "BIC"
PAIRS = [("VCB", "VHM"), ("HPG", "FPT")]               # cặp tương quan động theo đề
VAR_LEVELS = [0.95, 0.99]    # mức tin cậy VaR
HS_WINDOW = 250              # cửa sổ Historical Simulation (≈ 1 năm)
BASEL_WINDOW = 250           # cửa sổ Basel Traffic Light (250 phiên gần nhất)
VIOL_MODEL = "Normal"        # >>> MÔ HÌNH ĐÁNH DẤU VI PHẠM TRÊN MỌI HÌNH & DIỄN GIẢI TV4: "Normal" hoặc "t" <<<
VIOL_OOS = {"Normal": "DCC-N", "t": "DCC-t"}[VIOL_MODEL]   # tên tương ứng trong phần ngoài mẫu
EULER_CL = 0.99              # mức tin cậy dùng cho phân bổ rủi ro Euler
STRESS = {                   # giai đoạn khủng hoảng theo đề (TV5)
    "Covid-19 (02–04/2020)": ("2020-02-01", "2020-04-30"),
    "Trái phiếu & lãi suất (04–11/2022)": ("2022-04-01", "2022-11-30"),
}


STRESS_COLORS = ["tab:red", "tab:orange"]   # mỗi giai đoạn một màu để phân biệt Covid-19 với Trái phiếu & lãi suất


def shade_stress(ax, alpha=0.12, label=True, xlim=None):
    """Tô màu từng giai đoạn khủng hoảng (mỗi giai đoạn một màu, chú thích theo tên giai đoạn).
    xlim=(đầu, cuối): bỏ qua giai đoạn nằm hoàn toàn ngoài khoảng này (tránh kéo giãn trục thời gian)."""
    for (name, (s, e)), c in zip(STRESS.items(), STRESS_COLORS):
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        if xlim is not None and (e < xlim[0] or s > xlim[1]):
            continue
        ax.axvspan(s, e, color=c, alpha=alpha, label=(name if label else None))


def grid_axes(n, ncol=2, h=3.2, w=13, **kw):
    """Tạo lưới ô vẽ cho n mã, ẩn các ô thừa."""
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w, h * nrow), squeeze=False, **kw)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


print("BASE_DIR  :", BASE_DIR)
print("INPUT_DIR :", INPUT_DIR, "| tồn tại:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


def period_masks(index):
    """Mặt nạ True/False cho từng giai đoạn: các giai đoạn STRESS + 'Bình thường' (ngoài mọi giai đoạn stress)."""
    m = {name: (index >= s) & (index <= e) for name, (s, e) in STRESS.items()}
    m = {"Bình thường": ~np.any(np.column_stack(list(m.values())), axis=1), **m}
    return m


## Nạp kết quả của các thành viên trước

In [ ]:
import json
# ===================== NẠP KẾT QUẢ TV1 → TV4 =====================
def need(path, prev):
    """Báo lỗi rõ ràng nếu chưa chạy notebook của thành viên trước."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Chưa có {path} – hãy chạy notebook {prev} trước (xem README.md).")
    return Path(path)


log_ret = pd.read_csv(need(OUT["TV1"] / "log_returns.csv", "TV1"), index_col=0, parse_dates=True, float_precision="round_trip")
log_ret.index.name = "Date"
tickers = list(log_ret.columns); N = len(tickers); T = len(log_ret)
print(f"[TV1] log_ret: {T} phiên × {N} mã ({', '.join(tickers)}) | {log_ret.index.min().date()} -> {log_ret.index.max().date()}")
std_resid = pd.read_csv(need(OUT["TV2"] / "std_resid.csv", "TV2"), index_col=0, parse_dates=True, float_precision="round_trip")
cond_vol = pd.read_csv(need(OUT["TV2"] / "cond_vol.csv", "TV2"), index_col=0, parse_dates=True, float_precision="round_trip")
sel_h = pd.read_csv(need(OUT["TV2"] / "garch_selected_models.csv", "TV2"), index_col=0, float_precision="round_trip").loc[tickers]
best_name = sel_h["Mô hình"]                                                       # tên mô hình biên đã chọn
SPEC = {tk: dict(gjr=bool(sel_h.loc[tk, "gjr"]), dist=str(sel_h.loc[tk, "dist"])) for tk in tickers}
mu_vec = sel_h["mu"].astype(float)                                                  # μ thập phân
sigma_fcst = sel_h["sigma_T+1"].astype(float).rename("sigma_T+1")
selected = {tk: {"k": int(sel_h.loc[tk, "k"]), "converged": bool(sel_h.loc[tk, "converged"])} for tk in tickers}   # bản rút gọn
print("[TV2] mô hình biên:", best_name.to_dict())
z_dcc = np.load(need(OUT["TV3"] / "dcc_matrices.npz", "TV3"), allow_pickle=False)
assert list(pd.to_datetime(z_dcc["dates"])) == list(log_ret.index) and [str(t) for t in z_dcc["tickers"]] == tickers, \
    "Output TV1 và TV3 không khớp ngày/mã – chạy lại từ TV1."
R_t, H_t, R_fcst, H_fcst = z_dcc["R"], z_dcc["H"], z_dcc["R_fcst"], z_dcc["H_fcst"]
a_hat, b_hat = float(z_dcc["a"]), float(z_dcc["b"])
ok_stat = (a_hat >= 0) and (b_hat >= 0) and (a_hat + b_hat < 1)
S = cond_vol.values if "cond_vol" in globals() else np.sqrt(np.einsum("tii->ti", H_t))
idx_tk = {tk: i for i, tk in enumerate(tickers)}
masks = period_masks(log_ret.index)
iu_ = np.triu_indices(N, 1)
avg_corr = pd.Series(R_t[:, iu_[0], iu_[1]].mean(axis=1), index=log_ret.index)      # tương quan DCC trung bình các cặp
print(f"[TV3] H_t {H_t.shape} | a = {a_hat:.4f}, b = {b_hat:.4f}")

# ---- TV4 ----
with open(need(OUT["TV4"] / "tv4_settings.json", "TV4"), encoding="utf-8") as fh:
    cfg4 = json.load(fh)
WEIGHTS = pd.Series(cfg4["weights"]).loc[tickers]
w = WEIGHTS.values
N_TEST, REFIT = int(cfg4["N_TEST"]), int(cfg4["REFIT"])
r_p = pd.Series(log_ret.values @ w, index=log_ret.index, name="r_p")
sigma_p = pd.Series(np.sqrt(np.einsum("i,tij,j->t", w, H_t, w)), index=log_ret.index, name="sigma_p")
undiv = pd.Series(S @ w, index=log_ret.index)                                       # σ danh mục nếu mọi tương quan = 1
var_es = pd.read_excel(need(OUT["TV4"] / "TV4_var_backtest.xlsx", "TV4"), sheet_name="VaR_ES_hang_ngay", index_col=0)
var_es.index = pd.to_datetime(var_es.index)
bt = var_es.iloc[HS_WINDOW:].copy()                                                 # cửa sổ backtest chung (như TV4)
bt_table = pd.read_excel(OUT["TV4"] / "TV4_var_backtest.xlsx", sheet_name="Backtest")
oos = pd.read_csv(need(OUT["TV4"] / "oos_var_es.csv", "TV4"), index_col=0, parse_dates=True, float_precision="round_trip")
oos_bt = pd.read_csv(need(OUT["TV4"] / "oos_backtest.csv", "TV4"))
print(f"[TV4] var_es {var_es.shape}, bt {len(bt)} phiên, oos {len(oos)} phiên | w = {WEIGHTS.round(3).to_dict()}")

---
# THÀNH VIÊN 5: PHÂN BỔ RỦI RO EULER, STRESS TESTING & TÍCH HỢP CODE
## 5.0 Kiểm tra tích hợp (Code Integration)
Notebook này đã chạy theo đúng trình tự TV1 → TV2 → TV3 → TV4 với **các biến dùng chung thống nhất**:
`log_ret` (TV1) → `std_resid`, `cond_vol`, `mu_vec`, `sigma_fcst` (TV2) → `R_t`, `H_t`, `H_fcst` (TV3) → `sigma_p`, `var_es`, `bt_table`, `oos`, `oos_bt` (TV4).
Ô dưới kiểm tra tự động tính tương thích; mọi hạng mục phải "Đạt".

In [ ]:
# ===================== 5.0 KIỂM TRA TÍCH HỢP =====================
eig_min_all = np.linalg.eigvalsh(H_t).min()
checks = [
    ("Cùng danh sách mã TV1/TV2/TV3", "TV5", list(log_ret.columns) == list(std_resid.columns) == list(cond_vol.columns) == tickers, ", ".join(tickers)),
    ("Cùng lịch ngày TV1/TV2/TV4", "TV5", log_ret.index.equals(std_resid.index) and log_ret.index.equals(var_es.index),
     f"{len(log_ret)} phiên {log_ret.index[0].date()} → {log_ret.index[-1].date()}"),
    ("H_t có kích thước T × N × N", "TV5", H_t.shape == (T, N, N), str(H_t.shape)),
    ("Không có NaN trong r, ε, σ, H", "TV5", not (log_ret.isna().any().any() or std_resid.isna().any().any()
                                                  or cond_vol.isna().any().any() or np.isnan(H_t).any()), ""),
    ("sqrt(diag H_t) = σ_t (TV2 ↔ TV3)", "TV5", np.allclose(np.sqrt(np.einsum("tii->ti", H_t)), cond_vol.values), ""),
    ("H_t đối xứng & xác định dương", "TV5", np.allclose(H_t, H_t.transpose(0, 2, 1)) and eig_min_all > 0, f"λ_min = {eig_min_all:.2e}"),
    ("DCC dừng: a ≥ 0, b ≥ 0, a + b < 1", "TV5", bool(ok_stat), f"a + b = {a_hat + b_hat:.4f}"),
    ("σ_p TV4 = sqrt(w'H_t w) với cùng w", "TV5", np.allclose(sigma_p.values, np.sqrt(np.einsum("i,tij,j->t", w, H_t, w))),
     "w = " + str({tk: round(float(x), 4) for tk, x in zip(tickers, w)})),
    ("Tổng tỷ trọng = 1", "TV5", np.isclose(w.sum(), 1), ""),
    ("Mọi GARCH đơn biến hội tụ", "TV5", all(f["converged"] for f in selected.values()), ""),
    ("VaR/ES ngoài mẫu: đủ N_TEST ngày, không NaN, cùng lịch với var_es", "TV5",
     len(oos) == N_TEST and not oos.isna().any().any() and oos.index.isin(var_es.index).all(), f"{len(oos)} phiên"),
]
integration = pd.DataFrame(checks, columns=["Hạng mục", "Thành viên", "Đạt", "Ghi chú"])
display(integration)
integration.to_csv(OUT["TV5"] / "integration_check.csv", index=False, encoding="utf-8-sig")
print("✔ Toàn bộ pipeline TV1 → TV4 tương thích." if integration["Đạt"].all() else "✘ CÓ HẠNG MỤC KHÔNG ĐẠT – xem bảng trên.")

## 5.1 Đóng góp rủi ro Euler (Euler Risk Contributions)
**Tầng 1:**
- a. Marginal VaR: $MC\text{-}VaR_i = \dfrac{\partial VaR_p}{\partial w_i} = z_{1-\alpha}\dfrac{(H_tw)_i}{\sigma_{p,t}}$.
- b. Euler contribution: $RC_i = w_i \times MC\text{-}VaR_i$.
- c. Tính cộng Euler: $\sum_i RC_i = VaR_p$ (vì VaR thuần nhất bậc 1 theo w).
- d. $\%RC_i = RC_i / VaR_p \times 100\%$.

**Tầng 2:** Input `H_t`, `H_fcst`, `w`; mức tin cậy `EULER_CL` (mặc định 99%). Output `rc_ts` (T × N, đóng góp % theo ngày), bảng T+1.

**Tầng 3:** %RC_i > w_i ⇒ mã **tập trung rủi ro** (đóng góp nhiều hơn phần vốn); %RC_i < w_i ⇒ mã có tác dụng đa dạng hoá.

In [ ]:
# ===================== 5.1 EULER RC =====================
z_cl = stats.norm.ppf(EULER_CL)


def euler(H, w):
    """Trả về VaR_p, MC-VaR (N), RC (N) cho một ma trận H (N×N)."""
    sp = np.sqrt(w @ H @ w)
    mc = z_cl * (H @ w) / sp
    rc = w * mc
    return z_cl * sp, mc, rc


Hw = np.einsum("tij,j->ti", H_t, w)
VaR_p_ts = z_cl * sigma_p.values
MC_ts = z_cl * Hw / sigma_p.values[:, None]
RC_ts = w * MC_ts
pct_rc = pd.DataFrame(RC_ts / VaR_p_ts[:, None] * 100, index=log_ret.index, columns=tickers)
err = np.abs(RC_ts.sum(axis=1) - VaR_p_ts).max()
print(f"Kiểm chứng tính cộng Euler: max_t |Σ RC_i − VaR_p| = {err:.2e} -> {'ĐẠT' if err < 1e-10 else 'KHÔNG ĐẠT'}")
print(f"Khớp VaR Normal {int(EULER_CL * 100)}% của TV4: "
      f"{np.allclose(VaR_p_ts, var_es[f'VaR_Normal_{int(EULER_CL * 100)}'].values)}")

VaR_f, MC_f, RC_f = euler(H_fcst, w)
tbl_fcst = pd.DataFrame({"w_i (%)": w * 100, "σ_i T+1 (%/ngày)": sigma_fcst.values * 100, "MC-VaR_i": MC_f,
                         "RC_i (%)": RC_f * 100, "%RC_i": RC_f / VaR_f * 100}, index=tickers)
tbl_fcst["%RC_i − w_i (điểm %)"] = tbl_fcst["%RC_i"] - tbl_fcst["w_i (%)"]
tbl_fcst.loc["TỔNG"] = [100, np.nan, np.nan, RC_f.sum() * 100, 100, 0]
print(f"\nDự báo phiên T+1 (sau {log_ret.index[-1].date()}): VaR_p {int(EULER_CL * 100)}% = {VaR_f:.4%} | Σ RC_i = {RC_f.sum():.4%}")
display(tbl_fcst.round(3))

# ---- Xuất chuỗi hàng ngày đầy đủ: MC-VaR, RC, %RC, VaR_p (giữ định dạng file cũ) ----
mc_df = pd.DataFrame(MC_ts, index=log_ret.index, columns=tickers)
rc_df = pd.DataFrame(RC_ts, index=log_ret.index, columns=tickers)
euler_daily = pd.concat({"MC_VaR": mc_df, "RC": rc_df, "PctRC": pct_rc / 100,
                         "Portfolio": pd.DataFrame({"VaR_p": VaR_p_ts}, index=log_ret.index)}, axis=1)
euler_daily.index.name = "Date"
euler_daily.to_csv(OUT["TV5"] / "euler_rc_timeseries.csv", encoding="utf-8-sig")

# ---- Hình: %RC làm mượt 5 ngày (bớt nhiễu); nét đứt trắng = tỷ trọng vốn tích luỹ ----
pct_sm = pct_rc.rolling(5, min_periods=1).mean()
fig, axes = plt.subplots(2, 1, figsize=(13, 7.5), sharex=True, gridspec_kw={"height_ratios": [2.6, 1]})
axes[0].stackplot(pct_sm.index, pct_sm.T.values, labels=tickers, alpha=0.85)
for c_ in np.cumsum(w)[:-1] * 100:
    axes[0].axhline(c_, color="white", ls="--", lw=0.7)
axes[0].set_ylim(0, 100); axes[0].set_ylabel("%RC (trung bình 5 ngày)")
axes[0].set_title(f"Tỷ lệ đóng góp rủi ro Euler theo thời gian (VaR {int(EULER_CL * 100)}%) – nét đứt trắng = tỷ trọng vốn")
axes[0].legend(ncol=N, fontsize=8, loc="upper left")
axes[1].plot(log_ret.index, VaR_p_ts * 100, color="tab:red", lw=1)
shade_stress(axes[1], alpha=0.15)
axes[1].set_ylabel("VaR_p (%)"); axes[1].grid(alpha=0.3); axes[1].legend(fontsize=7, loc="upper right")
plt.tight_layout(); plt.savefig(OUT["TV5"] / "5_euler_rc_timeseries.png", dpi=150); plt.show()


## 5.2 Phân tích Stress Periods
**Tầng 1:** So sánh giai đoạn bình thường với (1) Covid-19, 02–04/2020 và (2) khủng hoảng trái phiếu doanh nghiệp, thanh khoản & lãi suất, 04–11/2022.
Hệ số đa dạng hoá $DR = \sum_i w_i\sigma_{i,t}/\sigma_{p,t}$ (DR = 1: không còn đa dạng hoá); lợi ích đa dạng hoá $= 1 - 1/DR$.

**Tầng 3:** đọc các câu hỏi của đề ngay trong phần in ra: mã đóng góp lớn nhất, mã tăng đóng góp mạnh nhất, tương quan thay đổi ra sao
và ảnh hưởng tới tổng rủi ro.

In [ ]:
# ===================== 5.2 STRESS PERIODS =====================
ann = np.sqrt(TRADING_DAYS)
hits99 = (-var_es["r_p"] > var_es[f"VaR_Normal_{int(EULER_CL * 100)}"]).where(var_es.index >= bt.index[0])
DR = undiv / sigma_p
stress_summary = pd.DataFrame({name: {
    "Số phiên": int(m.sum()),
    "Lợi suất DM tích luỹ cả giai đoạn (%)": (np.exp(r_p[m].sum()) - 1) * 100,   # thay cho số năm hoá dễ gây hiểu nhầm
    "Lợi suất DM TB ngày (%)": r_p[m].mean() * 100,
    "Lợi suất DM TB năm hoá (%) – tham khảo": r_p[m].mean() * TRADING_DAYS * 100,
    "σ_p năm hoá (%)": sigma_p[m].mean() * ann * 100,
    f"VaR_p {int(EULER_CL * 100)}% TB (%)": VaR_p_ts[m].mean() * 100,
    "Lỗ lớn nhất 1 ngày (%)": -r_p[m].min() * 100,
    "Tương quan TB các cặp": avg_corr[m].mean(),
    "Hệ số đa dạng hoá DR": DR[m].mean(),
    "Lợi ích đa dạng hoá 1 − 1/DR (%)": (1 - 1 / DR[m]).mean() * 100,
    f"Tỷ lệ vi phạm VaR-N {int(EULER_CL * 100)}% (%)": hits99[m].mean() * 100,
} for name, m in masks.items()})
print("Tổng quan danh mục theo giai đoạn (lợi suất tích luỹ phụ thuộc độ dài giai đoạn – so sánh bằng dòng TB ngày):"); display(stress_summary.round(3))

prc_period = pd.DataFrame({name: pct_rc[m].mean() for name, m in masks.items()})
prc_period.insert(0, "Tỷ trọng w (%)", w * 100)
mc_period = pd.DataFrame({name: mc_df[m].mean() for name, m in masks.items()})     # MC-VaR trung bình theo giai đoạn
for name in STRESS:
    prc_period[f"Δ%RC {name} (điểm %)"] = prc_period[name] - prc_period["Bình thường"]
print("Tỷ lệ đóng góp rủi ro %RC_i trung bình theo giai đoạn:"); display(prc_period.round(2))
print("MC-VaR_i trung bình theo giai đoạn (thập phân):"); display(mc_period.round(4))

for name in STRESS:
    m = masks[name]
    top = prc_period[name].idxmax()
    up = prc_period[f"Δ%RC {name} (điểm %)"].idxmax()
    dn = prc_period[f"Δ%RC {name} (điểm %)"].idxmin()
    print(f"\n[{name}]")
    print(f"  • Đóng góp rủi ro lớn nhất: {top} ({prc_period.loc[top, name]:.1f}% VaR, tỷ trọng {w[idx_tk[top]] * 100:.0f}%).")
    print(f"  • Tăng đóng góp mạnh nhất so với bình thường: {up} ({prc_period.loc[up, f'Δ%RC {name} (điểm %)']:+.1f} điểm %); "
          f"giảm mạnh nhất: {dn} ({prc_period.loc[dn, f'Δ%RC {name} (điểm %)']:+.1f} điểm %).")
    print(f"  • σ_p năm hoá: {stress_summary.loc['σ_p năm hoá (%)', 'Bình thường']:.1f}% → {stress_summary.loc['σ_p năm hoá (%)', name]:.1f}%; "
          f"tương quan TB: {stress_summary.loc['Tương quan TB các cặp', 'Bình thường']:.3f} → {stress_summary.loc['Tương quan TB các cặp', name]:.3f}; "
          f"lợi ích đa dạng hoá: {stress_summary.loc['Lợi ích đa dạng hoá 1 − 1/DR (%)', 'Bình thường']:.1f}% → "
          f"{stress_summary.loc['Lợi ích đa dạng hoá 1 − 1/DR (%)', name]:.1f}%.")

cols_p = ["Bình thường"] + list(STRESS)
fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.8))
xpos, bw = np.arange(N), 0.2
axes[0].bar(xpos - 1.5 * bw, w * 100, bw, color="lightgrey", label="Tỷ trọng w")
for j, (cn, col) in enumerate(zip(cols_p, ["tab:blue", "tab:orange", "tab:green"])):
    axes[0].bar(xpos + (j - 0.5) * bw, prc_period[cn].values, bw, color=col, label=cn)
axes[0].set_xticks(xpos); axes[0].set_xticklabels(tickers); axes[0].set_ylabel("%")
axes[0].set_title("Tỷ trọng vốn vs %RC theo giai đoạn"); axes[0].grid(alpha=0.3, axis="y")
axes[0].set_ylim(0, max(prc_period[cols_p].values.max(), w.max() * 100) * 1.3)
axes[0].legend(fontsize=7, loc="upper right")
l1, = axes[1].plot(avg_corr.index, avg_corr, color="black", lw=1.1, label="Tương quan TB (trái)")
ax2 = axes[1].twinx()
l2, = ax2.plot(DR.index, (1 - 1 / DR) * 100, color="tab:green", lw=0.9, label="Lợi ích đa dạng hoá % (phải)")
shade_stress(axes[1], alpha=0.15)
h_sh, l_sh = axes[1].get_legend_handles_labels()
axes[1].set_title("Correlation breakdown & lợi ích đa dạng hoá"); axes[1].grid(alpha=0.3)
axes[1].legend(h_sh + [l2], l_sh + [l2.get_label()], fontsize=7, loc="lower left")
plt.tight_layout(); plt.savefig(OUT["TV5"] / "5_stress_periods.png", dpi=150); plt.show()


### 5.2b Kịch bản tương quan (Correlation Breakdown)
Giữ nguyên biến động dự báo hiện tại $\sigma_{T+1}$, thay ma trận tương quan bằng tương quan trung bình của từng giai đoạn và trường hợp xấu nhất ρ = 1
⇒ tách riêng tác động của **tương quan** lên VaR và %RC.

In [ ]:
# ===================== 5.2b KỊCH BẢN TƯƠNG QUAN =====================
scen = {"ρ dự báo T+1 (DCC)": R_fcst}
scen.update({f"ρ TB {name}": R_t[m].mean(axis=0) for name, m in masks.items()})
scen["ρ = 1 (tệ nhất)"] = np.ones((N, N))
rows = {}
for name, R in scen.items():
    Hs = R * np.outer(sigma_fcst.values, sigma_fcst.values)
    v, _, rc = euler(Hs, w)
    rows[name] = {"VaR_p (%)": v * 100, **{f"%RC {tk}": rc[i] / v * 100 for i, tk in enumerate(tickers)}}
corr_scen = pd.DataFrame(rows).T
display(corr_scen.round(3))
base_var = corr_scen.iloc[0, 0]
for k in corr_scen.index[1:]:
    v = corr_scen.loc[k, "VaR_p (%)"]
    print(f"• Với {k}: VaR_p = {v:.2f}% ({(v / base_var - 1):+.1%} so với dự báo DCC {base_var:.2f}%)")

### 5.2c Danh mục cân bằng rủi ro (Equal Risk Contribution) – tham khảo cho khuyến nghị
Tìm w sao cho mọi mã đóng góp như nhau: $RC_i = VaR_p/N$ (thuật toán lặp đơn giản trên $H_{T+1}$).

In [ ]:
# ===================== 5.2c RISK PARITY (ERC) =====================
def risk_parity(Hm, n_iter=1000, tol=1e-12):
    wv = np.ones(len(Hm)) / len(Hm)
    for _ in range(n_iter):
        mrc = Hm @ wv
        new = 1 / mrc
        new /= new.sum()
        new = 0.5 * wv + 0.5 * new                      # bước giảm chấn cho hội tụ ổn định
        if np.abs(new - wv).max() < tol:
            break
        wv = new
    return wv


w_erc = risk_parity(H_fcst)
v_erc, _, rc_erc = euler(H_fcst, w_erc)
rp_table = pd.DataFrame({"w hiện tại (%)": w * 100, "%RC hiện tại": RC_f / VaR_f * 100,
                         "w ERC (%)": w_erc * 100, "%RC ERC": rc_erc / v_erc * 100}, index=tickers)
for name, m in masks.items():                           # ERC theo H_t trung bình của từng giai đoạn (stress)
    if name != "Bình thường":
        rp_table[f"ERC theo H TB {name} (%)"] = risk_parity(H_t[m].mean(axis=0)) * 100
display(rp_table.round(2))
print(f"VaR_p {int(EULER_CL * 100)}% (T+1): hiện tại = {VaR_f:.3%} | danh mục ERC = {v_erc:.3%}")

## 5.3 Diễn giải cốt lõi & Khuyến nghị quản trị rủi ro

In [ ]:
# ===================== 5.3 KHUYẾN NGHỊ TỰ ĐỘNG =====================
over = tbl_fcst.drop("TỔNG")["%RC_i − w_i (điểm %)"].sort_values(ascending=False)
lines = ["KHUYẾN NGHỊ QUẢN TRỊ RỦI RO (sinh tự động từ kết quả – cần biên tập khi viết báo cáo)", "=" * 80,
         "1. Tập trung rủi ro hiện tại (T+1): " + ", ".join(
             f"{tk} gánh {tbl_fcst.loc[tk, '%RC_i']:.1f}% rủi ro với {tbl_fcst.loc[tk, 'w_i (%)']:.0f}% vốn"
             for tk in over.index[over > 0]) + " -> ứng viên GIẢM tỷ trọng.",
         "   Mã đóng góp rủi ro thấp hơn tỷ trọng (đa dạng hoá tốt): " + ", ".join(over[over <= 0].sort_values().index) + "."]
for i, name in enumerate(STRESS, start=1):
    s = stress_summary[name]; nrm = stress_summary["Bình thường"]
    up = prc_period[f"Δ%RC {name} (điểm %)"].idxmax()
    lines.append(f"2.{i} {name}: σ_p tăng {s['σ_p năm hoá (%)'] / nrm['σ_p năm hoá (%)']:.2f} lần, tương quan TB "
                 f"{nrm['Tương quan TB các cặp']:.2f} → {s['Tương quan TB các cặp']:.2f}, lợi ích đa dạng hoá "
                 f"{nrm['Lợi ích đa dạng hoá 1 − 1/DR (%)']:.1f}% → {s['Lợi ích đa dạng hoá 1 − 1/DR (%)']:.1f}%; "
                 f"{up} tăng đóng góp mạnh nhất ({prc_period.loc[up, f'Δ%RC {name} (điểm %)']:+.1f} điểm %).")
lines += [
    "3. Khi thị trường vào giai đoạn căng thẳng (σ_p hoặc tương quan TB vượt ngưỡng lịch sử): giảm tỷ trọng các mã có %RC − w lớn nhất,"
    " chuyển dần về danh mục ERC (bảng risk parity), và/hoặc giảm TỔNG mức đầu tư thay vì chỉ hoán đổi mã, vì tương quan tăng làm đa dạng hoá giữa cổ phiếu trong nước kém hiệu quả.",
    "4. Đa dạng hoá ngành: nên duy trì (mã khác ngành có tương quan thấp hơn), NHƯNG lợi ích này suy giảm trong khủng hoảng (DR giảm) -> "
    "cần bổ sung tài sản ít tương quan với cổ phiếu (trái phiếu chính phủ, tiền gửi, vàng) cho danh mục lớn.",
    "5. Dùng Euler RC như hạn mức: đặt trần %RC_i (ví dụ ≤ 30% VaR) và cảnh báo sớm khi %RC_i − w_i > 5 điểm %, theo dõi hằng ngày bằng H_t từ DCC; "
    "hạn mức VaR theo mã = %RC_i × VaR tổng; vượt trần -> tái cân bằng. Hiện tại vượt ngưỡng cảnh báo 5 điểm %: "
    + (", ".join(over.index[over > 5]) or "không có") + ".",
    f"6. Backtest TV4 (trong mẫu, VaR 99%): " + "; ".join(f"{r['Phương pháp']}: {r['Vi phạm']}/{r['Kỳ vọng']:.0f} vi phạm ({r['Kết luận']})" for _, r in bt_table.iterrows()
                                                         if r["Mức tin cậy"] == "99%") + ". "
    + "Ngoài mẫu (ước lượng lại mỗi {} ngày): ".format(REFIT)
    + "; ".join(f"VaR {lv}: tốt nhất theo quantile loss = "
                f"{oos_bt[oos_bt['Mức tin cậy'] == lv].set_index('Phương pháp')['Quantile loss (x1e4)'].idxmin()}, "
                f"mô hình bị đánh dấu thổi phồng/đánh giá thấp = "
                f"{', '.join(oos_bt[(oos_bt['Mức tin cậy'] == lv) & oos_bt['Đánh giá'].str.contains('THẤP|THỔI PHỒNG')]['Phương pháp']) or 'không có'}"
                for lv in ['95%', '99%'])
    + ". Chọn mô hình báo cáo dựa trên kết quả ngoài mẫu trước khi dùng cho tính vốn.",
]
rec_text = "\n".join(lines)
print(rec_text)
(OUT["TV5"] / "recommendations.txt").write_text(rec_text, encoding="utf-8")

with pd.ExcelWriter(OUT["TV5"] / "TV5_euler_stress.xlsx", engine="openpyxl") as xw:
    integration.to_excel(xw, sheet_name="Kiem_tra_tich_hop", index=False)
    tbl_fcst.to_excel(xw, sheet_name="Euler_T+1")
    stress_summary.to_excel(xw, sheet_name="Stress_tong_quan")
    prc_period.to_excel(xw, sheet_name="PctRC_theo_giai_doan")
    corr_scen.to_excel(xw, sheet_name="Kich_ban_tuong_quan")
    rp_table.to_excel(xw, sheet_name="Risk_parity")
    mc_period.to_excel(xw, sheet_name="MC_VaR_theo_giai_doan")
    pct_rc.to_excel(xw, sheet_name="PctRC_hang_ngay")
print(f"\n✔ HOÀN TẤT TOÀN BỘ PIPELINE. Kết quả nằm trong: {ROOT_OUT}")

### TV5 – Gợi ý trả lời vấn đáp
**Câu 1 – Vì sao mã có tỷ trọng nhỏ vẫn có thể đóng góp rủi ro lớn nhất?** $RC_i = w_i \cdot z\,(H_tw)_i/\sigma_p$ phụ thuộc không chỉ $w_i$ mà còn
**biến động riêng** $\sigma_i$ và **hiệp phương sai với phần còn lại của danh mục** $(H_tw)_i = \sum_j w_j\rho_{ij}\sigma_i\sigma_j$. Một mã rất biến
động và tương quan cao với các mã khác có MC-VaR lớn, nên dù w nhỏ vẫn có thể có RC lớn nhất; ngược lại mã có tương quan thấp/âm đóng góp ít,
thậm chí âm (phòng hộ).

**Câu 2 – Correlation breakdown ảnh hưởng Euler contribution thế nào?** Khi tương quan tăng vọt, $(H_tw)_i$ của mọi mã tăng ⇒ VaR danh mục tăng
(bảng kịch bản: ρ = 1 là trường hợp tệ nhất, VaR = tổng VaR từng mã) và lợi ích đa dạng hoá biến mất. Đóng góp rủi ro dịch chuyển về các mã
**biến động cao nhất**, vì khi ρ → 1 thì %RC_i → $w_i\sigma_i/\sum_j w_j\sigma_j$. Những mã vốn đóng vai trò đa dạng hoá (tương quan thấp lúc bình thường)
mất tác dụng ⇒ hạn mức rủi ro tính trong giai đoạn bình thường sẽ bị vượt.